这是**熵权法 (Entropy Weight Method, EWM)** 的详细解析。

如果不喜欢 AHP 那种“拍脑袋”的主观打分，或者你手里已经有了具体的**数据表格**（比如各省份的GDP、人口、绿化率等具体的数字），那么熵权法是给指标赋权的最好选择。它是**客观赋权法**的代表。

---

### 一、 算法原理
**核心逻辑**：**数据差异越大，权重越高。**

*   **信息熵**是一个物理/信息论概念，用来衡量混乱程度。
*   **直观理解**：
    *   如果是**考试**：
        *   数学卷子：大家考分从 10分 到 100分 都有，拉开了差距。说明数学这门课**区分度高**，含金量（信息量）大，应该给**高权重**。
        *   体育卷子：大家都得 90-95 分，几乎没区别。说明体育这门课**区分度低**，信息量少，应该给**低权重**。
*   **结论**：某个指标的一列数据，如果方差大（波动大），说明它蕴含的信息多，权重就应该大；反之，若数值都差不多，权重就小。

---

### 二、 推导与步骤

假设有 $n$ 个样本（行），$m$ 个指标（列），数据矩阵为 $X$。

#### 1. 数据标准化 (非常关键)
因为各指标单位不同（GDP是万亿，人口是人，比例是%），必须统一。同时要区分**正向指标**和**负向指标**。

*   **正向指标**（越大越好，如收入）：
    $$Y_{ij} = \frac{X_{ij} - \min(X_j)}{\max(X_j) - \min(X_j)}$$
*   **负向指标**（越小越好，如污染浓度）：
    $$Y_{ij} = \frac{\max(X_j) - X_{ij}}{\max(X_j) - \min(X_j)}$$

#### 2. 计算比重 (概率矩阵)
计算第 $j$ 项指标下，第 $i$ 个样本所占的比重 $P_{ij}$：
$$P_{ij} = \frac{Y_{ij}}{\sum_{i=1}^{n} Y_{ij}}$$

#### 3. 计算信息熵
计算第 $j$ 项指标的信息熵 $E_j$：
$$E_j = -k \sum_{i=1}^{n} P_{ij} \ln(P_{ij})$$
其中，常数 $k = \frac{1}{\ln(n)}$。
*(注意：如果 $P_{ij}=0$，则 $\ln(P_{ij})$ 无意义，计算时通常加一个极小值，如 $10^{-6}$)*

#### 4. 计算信息效用值 (差异系数)
$$D_j = 1 - E_j$$
信息熵 $E_j$ 越大（越接近1），说明数据越乱（越无序/越均匀），信息效用 $D_j$ 就越小。

#### 5. 计算最终权重
归一化差异系数，得到权重 $W_j$：
$$W_j = \frac{D_j}{\sum_{j=1}^{m} D_j}$$

---

### 三、 适用场景
1.  **客观评价**：完全基于数据说话，没有任何人为偏好。
2.  **数据支撑充足**：必须要有完整的指标数据表（不能有缺失值，有的话先补全）。
3.  **常与TOPSIS连用**：先用熵权法算出权重，再用TOPSIS法进行综合评价排序（即“基于熵权法的TOPSIS模型”）。

---

### 四、 Python 代码框架

这份代码设计得非常通用。你只需要传入一个 DataFrame，并告诉代码哪些列是**正向**的，哪些是**负向**的，它就会返回各指标的权重。

```python
import pandas as pd
import numpy as np

def entropy_weight_method(data, direction_list):
    """
    熵权法计算权重
    :param data: pandas DataFrame, 原始数据 (行=样本, 列=指标)
    :param direction_list: list, 指示每列是正向还是负向。
                           1 代表正向指标(越大越好), 0 代表负向指标(越小越好)
    :return: pandas Series, 各指标的权重
    """

    # 为了不破坏原数据，拷贝一份
    df = data.copy().astype(float)
    m_columns = df.columns
    n_rows = df.shape[0]

    # --- 1. 数据标准化 (Min-Max Scaling) ---
    # 这一步是为了消除量纲，并将数据映射到 [0, 1]
    for i, col in enumerate(m_columns):
        direction = direction_list[i]
        max_val = df[col].max()
        min_val = df[col].min()

        # 防止分母为0 (如果某列所有值都一样，max-min=0)
        if max_val == min_val:
            df[col] = 0.5 # 如果所有值一样，给予中间值，其实该指标权重最后会是0
        else:
            if direction == 1: # 正向指标
                df[col] = (df[col] - min_val) / (max_val - min_val)
            elif direction == 0: # 负向指标
                df[col] = (max_val - df[col]) / (max_val - min_val)
            else:
                raise ValueError(f"direction_list 中只能包含 0 或 1，位置 {i} 错误")

    # --- 2. 计算比重 P_ij ---
    # 有时候标准化后会出现0，导致后面 log(0) 报错。
    # 这里做一个平移，或者在计算log时加极小值。通常的做法是平移一丢丢，或者标准化范围设为 [0.001, 1]
    # 这里我们采用在 log 计算时加 epsilon 的方法，保持数据纯度

    # 计算每列的总和
    sum_cols = df.sum(axis=0)

    # 归一化得到 P_ij
    # 如果某列和为0（极端情况），则该列比重设为 1/n
    P = df / sum_cols
    P = P.fillna(1/n_rows) # 填充可能产生的NaN

    # --- 3. 计算熵值 E_j ---
    k = 1 / np.log(n_rows)
    # 加上 1e-6 避免 log(0) 导致 -inf
    E = -k * (P * np.log(P + 1e-6)).sum(axis=0)

    # --- 4. 计算差异系数 D_j ---
    D = 1 - E

    # --- 5. 计算权重 W_j ---
    # 如果所有指标的差异系数和为0（非常罕见），则平均分配权重
    if D.sum() == 0:
        W = pd.Series(np.ones(len(m_columns)) / len(m_columns), index=m_columns)
    else:
        W = D / D.sum()

    return W

# ================= 使用示例 =================

if __name__ == "__main__":
    # 1. 准备数据
    # 假设我们评价 3 个城市的 4 个指标
    # 指标: [GDP(正), 犯罪率(负), 绿化率(正), 交通拥堵指数(负)]
    data = {
        'GDP': [100, 120, 90],          # 越大越好
        '犯罪率': [5.2, 4.8, 6.0],       # 越小越好
        '绿化率': [30, 35, 25],          # 越大越好
        '拥堵指数': [8.5, 9.0, 7.5]      # 越小越好
    }
    df = pd.DataFrame(data, index=['城市A', '城市B', '城市C'])
    print("原始数据:\n", df)

    # 2. 定义指标方向 (1=正向, 0=负向)
    # 对应列顺序: GDP(1), 犯罪率(0), 绿化率(1), 拥堵指数(0)
    directions = [1, 0, 1, 0]

    # 3. 调用函数
    weights = entropy_weight_method(df, directions)

    # 4. 输出结果
    print("-" * 30)
    print("各指标权重 (熵权法):")
    print(weights)

    # (进阶) 计算各城市的综合得分
    # 简单的加权求和: Score = Norm_Data * Weights
    # 注意：这里需要用标准化后的数据乘权重，为了演示简单，我们重新简单标准化一下
    # 实际操作中，可以将 entropy_weight_method 函数修改为同时返回标准化数据

    print("-" * 30)
    print("权重最高的指标是:", weights.idxmax())
```

### 代码使用的“修修补补”指南：

1.  **输入数据 (DataFrame)**：
    *   确保数据里**全是数字**。如果你的Excel里有“1,000”这种带逗号的字符串，先在Excel里清理干净，或者用 `df = df.replace(',', '', regex=True).astype(float)` 处理。
    *   不能有 `NaN` (空值)。如果有，先用平均值填充 `df.fillna(df.mean())`。

2.  **指标方向 (`directions`)**：
    *   这是最容易出错的地方！一定要对着你的列名，一个个核对。
    *   **正向 (1)**：收入、GDP、成绩、优良率。
    *   **负向 (0)**：费用、污染、故障率、死亡率。

3.  **结果解读**：
    *   如果某个指标算出来的权重特别大（比如 0.6），而其他都很小，检查一下原始数据。
    *   这通常意味着这个指标在不同样本间**波动极其巨大**（例如：样本A是1，样本B是10000）。如果不希望权重这么极端，可能需要先对那一列取对数 `np.log()` 再传入模型，或者在论文中解释“该指标区分度极高”。